# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [ ]:
# importar librerías
import pandas as pd
import numpy as np

# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [ ]:
print('EXPLORACION DE DATASETS')
# cantidad de filas y columnas por dataset
filas_iniciales = len(orders)
print(f'Orders: {orders.shape} | Catalog: {catalog.shape} | Marketing: {marketing.shape}')

print('\nNULOS')
print(orders.isna().sum()[orders.isna().sum() > 0])
 
print('\nDUPLICADOS')
print('Filas duplicadas completas:', orders.duplicated().sum())
print('id_pedido repetidos:', orders['id_pedido'].duplicated().sum())
 
print('\nDISTRIBUCIÓN DE cantidad')
print(orders['cantidad'].value_counts(dropna=False).sort_index())
 
print('\nVALORES INVÁLIDOS EN MONTOS')
for col in ['precio_unitario', 'monto_descuento', 'monto_total']:
    print(f'{col}: negativos={(orders[col] < 0).sum()} | min={orders[col].min()} | max={orders[col].max()}')
 
print('\nCATEGÓRICAS INCONSISTENTES')
for col in ['pais', 'dispositivo', 'fuente_referencia', 'categoria_producto']:
    print(f'{col}: {sorted(orders[col].dropna().unique())}')
print('catalog.categoria_producto:', sorted(catalog['categoria_producto'].unique()))

EXPLORACION DE DATASETS
Orders: (25100, 12) | Catalog: (7, 4) | Marketing: (1620, 5)

NULOS
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     80
cantidad               50
precio_unitario        50
monto_descuento        50
dtype: int64

DUPLICADOS
Filas duplicadas completas: 100
id_pedido repetidos: 100

DISTRIBUCIÓN DE cantidad
-2.0            1
-1.0            3
 1.0        12394
 2.0        12642
 10000.0        6
 20000.0        4
 NaN           50
Name: cantidad, dtype: int64

VALORES INVÁLIDOS EN MONTOS
precio_unitario: negativos=0 | min=20.03 | max=499.96
monto_descuento: negativos=0 | min=0.0 | max=15.0
monto_total: negativos=4 | min=-492.65 | max=8840200.0

CATEGÓRICAS INCONSISTENTES
pais: ['Argentina', 'Colombia', 'Mexico', 'argentina', 'colombia', 'mexico']
dispositivo: ['desktop', 'mobile']
fuente_referencia: ['organic', 'paid_search', 'social']
categoria_producto: ['Electronica', 'Hogar', 'Moda']


### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [ ]:
# Limpieza de orders
print('LIMPIEZA ORDERS')
# Limpieza Duplicados
antes = len(orders)
orders = orders.drop_duplicates()
print(f'Duplicados eliminados: {antes - len(orders)}')
 
# Cambiar tipo de dato
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'])
orders['id_pedido'] = orders['id_pedido'].astype(str)
orders['id_usuario'] = orders['id_usuario'].astype(str)
 
#'mexico' y 'Mexico' son el mismo país
for col in ['pais', 'dispositivo', 'fuente_referencia', 'nombre_producto']:
    orders[col] = orders[col].str.strip()
    
# Poner los paises con mayuscula
orders['pais'] = orders['pais'].str.title()
orders['categoria_producto'] = orders['categoria_producto'].replace('Electrónica', 'Electronica')
 
print('Países corregidos:', sorted(orders['pais'].dropna().unique()))
 
# Se eliminaran las filas de cantidad > 10 unidades para mantener consistencia y no alterar revenue.
cond_alta = orders['cantidad'] > 100
# Valores de cantidad en negativos. Se mantienen pero se separaran del análisis
cond_neg = orders['cantidad'] < 0
# Se eliminaran filas con cantidad nula/0 -> registros incompletos. Esto debido a que la cantidad de nulos es minima
#ademas que no hay motivo para conservar una ventidad con cantidad 0. 
cond_nula = orders['cantidad'].isna() | (orders['cantidad'] == 0)
 
print(f'\nFilas con cantidad > 100: {cond_alta.sum()} '
      f'(aportaban ${orders.loc[cond_alta, "monto_total"].sum():,.2f} de revenue ficticio)')
print(f'Filas con cantidad negativa: {cond_neg.sum()}')
print(f'Filas con cantidad nula o cero: {cond_nula.sum()}')
 
# Se guardan aparte como evidencia en lugar de borrarlas sin dejar rastro
descartados = orders[cond_alta | cond_neg | cond_nula].copy()
descartados['motivo'] = np.select(
    [cond_alta, cond_neg],
    ['cantidad_fuera_de_rango', 'cantidad_negativa'],
    default='registro_incompleto'
)[(cond_alta | cond_neg | cond_nula).values]
 
orders = orders[~(cond_alta | cond_neg | cond_nula)].copy()
orders['cantidad'] = orders['cantidad'].astype(int)
 
# La categoría se recupera desde el catálogo usando el nombre del producto.
mapa_categoria = catalog.set_index('nombre_producto')['categoria_producto']
orders['categoria_producto'] = (orders['nombre_producto'].map(mapa_categoria)
                                .fillna(orders['categoria_producto']))
 
# Los nulos que no se puede recuperar se marca como Desconocido.
for col in ['pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto']:
    n = orders[col].isna().sum()
    if n:
        orders[col] = orders[col].fillna('Desconocido')
        print(f'Nulos marcados como Desconocido en {col}: {n}')
 
# --- 5.6 Validar consistencia de montos ---
esperado = orders['cantidad'] * orders['precio_unitario'] - orders['monto_descuento']
inconsistentes = (orders['monto_total'] - esperado).abs() > 0.05
print(f'\nFilas con monto_total inconsistente: {inconsistentes.sum()}')
 
print(f'\nFilas: {filas_iniciales} -> {len(orders)} ({len(orders)/filas_iniciales:.1%} conservado)')
orders.info()

LIMPIEZA ORDERS
Duplicados eliminados: 100
Países corregidos: ['Argentina', 'Colombia', 'Mexico']

Filas con cantidad > 100: 10 (aportaban $42,344,700.00 de revenue ficticio)
Filas con cantidad negativa: 4
Filas con cantidad nula o cero: 50
Nulos marcados como Desconocido en pais: 296
Nulos marcados como Desconocido en dispositivo: 20
Nulos marcados como Desconocido en fuente_referencia: 30
Nulos marcados como Desconocido en nombre_producto: 30
Nulos marcados como Desconocido en categoria_producto: 30

Filas con monto_total inconsistente: 0

Filas: 25100 -> 24936 (99.3% conservado)
<class 'pandas.core.frame.DataFrame'>
Int64Index: 24936 entries, 0 to 24999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           24936 non-null  object        
 1   id_usuario          24936 non-null  object        
 2   fecha_hora_pedido   24936 non-null  datetime64[ns]
 3   pais             

In [ ]:
# Incluir costo y proveedor en orders con left join
orders = orders.merge(
    catalog[['nombre_producto', 'costo_unitario', 'proveedor']],
    on='nombre_producto',
    how='left'
)
 
orders['costo_total'] = (orders['costo_unitario'] * orders['cantidad']).round(2)
orders['profit_bruto'] = (orders['monto_total'] - orders['costo_total']).round(2)
orders['margen_pct'] = (orders['profit_bruto'] / orders['monto_total'] * 100).round(2)
 
# Campos de apoyo para el dashboard
orders['fecha'] = orders['fecha_hora_pedido'].dt.normalize()
orders['mes'] = orders['fecha_hora_pedido'].dt.to_period('M').astype(str)
 
print('Pedidos sin costo (producto fuera del catálogo):', orders['costo_total'].isna().sum())
orders.head()

Pedidos sin costo (producto fuera del catálogo): 30


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,costo_unitario,proveedor,costo_total,profit_bruto,margen_pct,fecha,mes
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2,332.69,0.0,665.37,189.31,Mcmillan-Rhodes,378.62,286.75,43.10,2025-05-22,2025-05
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electrónica,1,176.86,5.0,171.86,25.21,Bowers LLC,25.21,146.65,85.33,2025-06-15,2025-06
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2,102.99,10.0,195.99,176.64,Long-Reid,353.28,-157.29,-80.25,2025-05-02,2025-05
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electrónica,1,257.87,15.0,242.87,25.21,Bowers LLC,25.21,217.66,89.62,2025-06-09,2025-06
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1,336.28,0.0,336.28,176.64,Long-Reid,176.64,159.64,47.47,2025-03-30,2025-03


In [ ]:
# Limpieza de catalog
# Unificar el acento: en orders la categoría viene sin acento y en catalog con acento.
catalog['categoria_producto'] = catalog['categoria_producto'].replace('Electrónica', 'Electronica')

# Quitar espacios innecesarios de nombre_producto.
catalog['nombre_producto'] = catalog['nombre_producto'].str.strip()

print('LIMPIEZA CATALOG')
print(catalog)
print('\nDuplicados en catalog:', catalog['nombre_producto'].duplicated().sum())
#No es necesario eliminar duplicados.

LIMPIEZA CATALOG
        nombre_producto categoria_producto  costo_unitario  \
0    Laptop-Gaming-16GB        Electronica          280.68   
1       Phone-Pro-128GB        Electronica           10.12   
2  Tablet-Standard-64GB        Electronica           25.21   
3        Blender-XL-Red              Hogar          176.64   
4      Vacuum-Pro-Black              Hogar           16.60   
5     Sneakers-Urban-42               Moda           17.21   
6       Jacket-Winter-M               Moda          189.31   

                 proveedor  
0   Fuller, Pena and Myers  
1                 King Ltd  
2               Bowers LLC  
3                Long-Reid  
4  Rivera, Carr and Finley  
5             Greene-Smith  
6          Mcmillan-Rhodes  

Duplicados en catalog: 0


In [ ]:
# Limpieza de marketing
print('LIMPIEZA MARKETING')
marketing['fecha'] = pd.to_datetime(marketing['fecha'])
 
# 'canal' se renombra a 'fuente_referencia' ya que describe la fuente de tráfico igual que en orders
if 'canal' in marketing.columns:
    marketing = marketing.rename(columns={'canal': 'fuente_referencia'})

# Se eliminan espacios innecesarios
marketing['pais'] = marketing['pais'].str.strip().str.title()

# Se eliminan duplicados
marketing = marketing.drop_duplicates().dropna()
 
print(marketing.info())
print('\nGasto total en marketing:', round(marketing['gasto'].sum(), 2))

LIMPIEZA MARKETING
<class 'pandas.core.frame.DataFrame'>
Int64Index: 1519 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   fecha              1519 non-null   datetime64[ns]
 1   pais               1519 non-null   object        
 2   id_campaña         1519 non-null   object        
 3   fuente_referencia  1519 non-null   object        
 4   gasto              1519 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 71.2+ KB
None

Gasto total en marketing: 2694664.43


---

In [ ]:
# preparar un nuevo archivo para KPI de Tableau
 
ventas_dia = (orders
              .groupby(['fecha', 'pais', 'fuente_referencia'])
              .agg(revenue=('monto_total', 'sum'),
                   costo_producto=('costo_total', 'sum'),
                   pedidos=('id_pedido', 'nunique'),
                   unidades=('cantidad', 'sum'))
              .reset_index())
 
mkt_dia = (marketing
           .groupby(['fecha', 'pais', 'fuente_referencia'])['gasto']
           .sum()
           .reset_index()
           .rename(columns={'gasto': 'gasto_marketing'}))
 
kpi_diario = ventas_dia.merge(mkt_dia, on=['fecha', 'pais', 'fuente_referencia'], how='outer').fillna(0)
kpi_diario['profit'] = (kpi_diario['revenue'] - kpi_diario['costo_producto'] - kpi_diario['gasto_marketing']).round(2)
kpi_diario['roas'] = np.where(kpi_diario['gasto_marketing'] > 0,
                              (kpi_diario['revenue'] / kpi_diario['gasto_marketing']).round(2),
                              np.nan)
 
# Validación: el revenue agregado debe coincidir con el de orders
print('Revenue orders:    ', round(orders['monto_total'].sum(), 2))
print('Revenue kpi_diario:', round(kpi_diario['revenue'].sum(), 2))
kpi_diario.head()

Revenue orders:     9622281.56
Revenue kpi_diario: 9622281.56


,fecha,pais,fuente_referencia,revenue,costo_producto,pedidos,unidades,gasto_marketing,profit,roas
0,2025-01-01,Argentina,organic,7499.05,2134.00,22,34,1365.62,3999.43,5.49
1,2025-01-01,Argentina,paid_search,4160.15,1815.80,12,17,1282.65,1061.70,3.24
2,2025-01-01,Argentina,social,3603.30,2012.56,12,16,1534.69,56.05,2.35
3,2025-01-01,Colombia,organic,6386.56,2454.57,16,26,2597.21,1334.78,2.46
4,2025-01-01,Colombia,paid_search,9047.47,3210.09,23,35,1771.40,4065.98,5.11


In [ ]:
# Reordenar las columnas en orders para mejor legibilidad.
columnas_orders = ['id_pedido', 'id_usuario', 'fecha_hora_pedido', 'fecha', 'mes', 'pais',
                   'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto',
                   'proveedor', 'cantidad', 'precio_unitario', 'monto_descuento', 'monto_total',
                   'costo_unitario', 'costo_total', 'profit_bruto', 'margen_pct']

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [ ]:
# exportar datasets
orders[columnas_orders].to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)
kpi_diario.to_csv('kpi_diario.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

---

In [ ]:
revenue_total = orders['monto_total'].sum()
costo_producto = orders['costo_total'].sum()
gasto_marketing = marketing['gasto'].sum()
costo_total = costo_producto + gasto_marketing
profit = revenue_total - costo_total
 
print('=' * 45)
print('RENTABILIDAD DEL NEGOCIO')
print('=' * 45)
print(f'Ingreso total (revenue):  ${revenue_total:>15,.2f}')
print(f'Costo de producto:        ${costo_producto:>15,.2f}')
print(f'Inversión en marketing:   ${gasto_marketing:>15,.2f}')
print(f'Costo total:              ${costo_total:>15,.2f}')
print('-' * 45)
print(f'PROFIT:                   ${profit:>15,.2f}')
print(f'Margen neto:              {profit / revenue_total * 100:>15.1f}%')
print(f'ROAS (revenue/marketing): {revenue_total / gasto_marketing:>15.2f}x')
print('=' * 45)
print(f'\n¿El negocio es rentable? {"SI" if profit > 0 else "NO"}')

RENTABILIDAD DEL NEGOCIO
Ingreso total (revenue):  $   9,622,281.56
Costo de producto:        $   3,828,869.01
Inversión en marketing:   $   2,694,664.43
Costo total:              $   6,523,533.44
---------------------------------------------
PROFIT:                   $   3,098,748.12
Margen neto:                         32.2%
ROAS (revenue/marketing):            3.57x

¿El negocio es rentable? SI


---

In [ ]:
# CELDA 2.2 — Comportamiento de ventas
# =============================================================
ticket_promedio = orders.groupby('id_pedido')['monto_total'].sum().mean()
cantidad_promedio = orders.groupby('id_pedido')['cantidad'].sum().mean()
total_pedidos = orders['id_pedido'].nunique()
total_usuarios = orders['id_usuario'].nunique()
 
print('=' * 55)
print('COMPORTAMIENTO DE VENTAS')
print('=' * 55)
print(f'Pedidos totales:                {total_pedidos:,}')
print(f'Usuarios que compraron:         {total_usuarios:,}')
print(f'Pedidos por usuario:            {total_pedidos / total_usuarios:.2f}')
print(f'Ticket promedio por orden:      ${ticket_promedio:,.2f}')
print(f'Productos promedio por orden:   {cantidad_promedio:.2f}')
print(f'Descuento promedio total:       ${orders["monto_descuento"].mean():,.2f}')
 
print('\nProductos más vendidos (unidades)')
mas_vendido = orders.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False)
print(mas_vendido.to_string())
print(f'\nProducto más vendido: {mas_vendido.index[0]} ({mas_vendido.iloc[0]:,} unidades)')
 
print('\nInversión en marketing por canal')
gasto_canal = marketing.groupby('fuente_referencia')['gasto'].sum().sort_values(ascending=False)
print(gasto_canal.to_string())

COMPORTAMIENTO DE VENTAS
Pedidos totales:                24,936
Usuarios que compraron:         7,640
Pedidos por usuario:            3.26
Ticket promedio por orden:      $385.88
Productos promedio por orden:   1.50
Descuento promedio total:       $4.50

Productos más vendidos (unidades)
nombre_producto
Vacuum-Pro-Black        6284
Blender-XL-Red          6279
Jacket-Winter-M         6256
Sneakers-Urban-42       6172
Laptop-Gaming-16GB      4198
Tablet-Standard-64GB    4153
Phone-Pro-128GB         4140
Desconocido               45

Producto más vendido: Vacuum-Pro-Black (6,284 unidades)

Inversión en marketing por canal
fuente_referencia
social         918043.21
organic        913533.01
paid_search    863088.21


---

In [ ]:
# Rentabilidad por producto

# El gasto de marketing no se puede repartir por producto: la inversión
# está registrada por día, país y canal, no por SKU. Por eso este corte
# usa profit BRUTO (revenue - costo de producto).
 
rent_producto = (orders
                 .groupby('nombre_producto')
                 .agg(unidades=('cantidad', 'sum'),
                      revenue=('monto_total', 'sum'),
                      costo=('costo_total', 'sum'),
                      profit_bruto=('profit_bruto', 'sum'),
                      costo_unitario=('costo_unitario', 'first'),
                      precio_promedio=('precio_unitario', 'mean'))
                 .round(2))
 
# Margen agregado total. No esta sumado por linea porque da un numero falso.
rent_producto['margen_%'] = (rent_producto['profit_bruto'] / rent_producto['revenue'] * 100).round(1)
 
# Los pedidos cuyo producto no está en el catálogo no tienen costo conocido.
# Mostrar su margen como 0% haría creer que no dejan ganancia, cuando en
# realidad no se puede calcular. Se marca como nulo.
rent_producto.loc[rent_producto['costo_unitario'].isna(), ['profit_bruto', 'margen_%']] = np.nan
rent_producto = rent_producto.sort_values('profit_bruto')
 
print('--- Rentabilidad por producto ---')
print(rent_producto.to_string())
 
perdedores = rent_producto[rent_producto['profit_bruto'] < 0]
if len(perdedores):
    print('\nProductos que pierden dinero:')
    for prod, fila in perdedores.iterrows():
        print(f'   {prod}: profit ${fila["profit_bruto"]:,.2f} | '
              f'costo unitario ${fila["costo_unitario"]:.2f} vs precio promedio ${fila["precio_promedio"]:.2f}')

--- Rentabilidad por producto ---
                      unidades     revenue       costo  profit_bruto  costo_unitario  precio_promedio  margen_%
nombre_producto                                                                                                
Laptop-Gaming-16GB        4198  1084672.98  1178294.64     -93621.66          280.68           261.27      -8.6
Jacket-Winter-M           6256  1606586.17  1184323.36     422262.81          189.31           259.82      26.3
Blender-XL-Red            6279  1611574.59  1109122.56     502452.03          176.64           259.16      31.2
Tablet-Standard-64GB      4153  1062922.80   104697.13     958225.67           25.21           257.53      90.2
Phone-Pro-128GB           4140  1057134.62    41896.80    1015237.82           10.12           259.53      96.0
Sneakers-Urban-42         6172  1566075.93   106220.12    1459855.81           17.21           257.23      93.2
Vacuum-Pro-Black          6284  1621051.85   104314.40    1516737.45  

In [ ]:
# Rentabilidad por categoría, país y canal

print('--- Por categoría (profit bruto) ---')
rent_categoria = (orders.groupby('categoria_producto')
                  .agg(revenue=('monto_total', 'sum'),
                       costo=('costo_total', 'sum'),
                       profit_bruto=('profit_bruto', 'sum'))
                  .round(2))
rent_categoria['margen_%'] = (rent_categoria['profit_bruto'] / rent_categoria['revenue'] * 100).round(1)
print(rent_categoria.sort_values('profit_bruto', ascending=False).to_string())
 
# Aquí sí se puede calcular profit NETO, porque kpi_diario tiene ventas y marketing al mismo grano (fecha + país + canal).

print('\n--- Por país (profit neto, incluye marketing) ---')
rent_pais = (kpi_diario.groupby('pais')
             .agg(revenue=('revenue', 'sum'),
                  costo_producto=('costo_producto', 'sum'),
                  marketing=('gasto_marketing', 'sum'),
                  profit=('profit', 'sum'))
             .round(2))
rent_pais['margen_%'] = (rent_pais['profit'] / rent_pais['revenue'] * 100).round(1)

# Sin inversión de marketing la división da infinito; se marca como nulo.

rent_pais['roas'] = (rent_pais['revenue'] / rent_pais['marketing']).replace([np.inf, -np.inf], np.nan).round(2) #retorno por cada peso
print(rent_pais.sort_values('profit', ascending=False).to_string())
print('\nNota: la fila "Desconocido" agrupa los pedidos sin país registrado.')
 
print('\n--- Por canal de adquisición (profit neto) ---')
rent_canal = (kpi_diario.groupby('fuente_referencia')
              .agg(revenue=('revenue', 'sum'),
                   costo_producto=('costo_producto', 'sum'),
                   marketing=('gasto_marketing', 'sum'),
                   profit=('profit', 'sum'),
                   pedidos=('pedidos', 'sum'))
              .round(2))

rent_canal['margen_%'] = (rent_canal['profit'] / rent_canal['revenue'] * 100).round(1)
rent_canal['roas'] = (rent_canal['revenue'] / rent_canal['marketing']).replace([np.inf, -np.inf], np.nan).round(2) #retorno por cada peso
rent_canal['cac'] = (rent_canal['marketing'] / rent_canal['pedidos']).round(2) #costo por cliente nuevo
print(rent_canal.sort_values('profit', ascending=False).to_string())

--- Por categoría (profit bruto) ---
                       revenue       costo  profit_bruto  margen_%
categoria_producto                                                
Hogar               3232626.44  1213436.96    2019189.48      62.5
Moda                3172662.10  1290543.48    1882118.62      59.3
Electrónica         3204730.40  1324888.57    1879841.83      58.7
Desconocido           12262.62        0.00          0.00       0.0

--- Por país (profit neto, incluye marketing) ---
                revenue  costo_producto  marketing      profit  margen_%  roas
pais                                                                          
Colombia     3173460.93      1270630.14  881667.78  1021163.01      32.2  3.60
Mexico       3211815.25      1278497.91  925775.48  1007541.86      31.4  3.47
Argentina    3125017.33      1236031.22  887221.17  1001764.94      32.1  3.52
Desconocido   111988.05        43709.74       0.00    68278.31      61.0   NaN

Nota: la fila "Desconocido" agrupa 

In [ ]:
# Evolución mensual

evolucion = (kpi_diario
             .assign(mes=kpi_diario['fecha'].dt.to_period('M').astype(str))
             .groupby('mes')
             .agg(revenue=('revenue', 'sum'),
                  costo_producto=('costo_producto', 'sum'),
                  marketing=('gasto_marketing', 'sum'),
                  profit=('profit', 'sum'),
                  pedidos=('pedidos', 'sum'))
             .round(2))

evolucion['margen_%'] = (evolucion['profit'] / evolucion['revenue'] * 100).round(1)
evolucion['revenue_mom_%'] = (evolucion['revenue'].pct_change() * 100).round(1)
evolucion['ticket_promedio'] = (evolucion['revenue'] / evolucion['pedidos']).round(2)
 
print('--- EVALUACION MENSUAL ---')
print(evolucion.to_string())
print('\nNota: los datos cubren de enero a junio de 2025 únicamente, por lo que no es posible calcular YoY. Se usa MoM.')

--- EVALUACION MENSUAL ---
            revenue  costo_producto  marketing     profit  pedidos  margen_%  revenue_mom_%  ticket_promedio
mes                                                                                                         
2025-01  1656274.48       661410.43  323007.50  671856.55     4316      40.6            NaN           383.75
2025-02  1442435.17       582461.21  448321.64  411652.32     3743      28.5          -12.9           385.37
2025-03  1657916.75       658068.92  516014.46  483833.37     4256      29.2           14.9           389.55
2025-04  1599342.28       645912.64  445179.82  508249.82     4215      31.8           -3.5           379.44
2025-05  1650821.10       640244.65  499181.34  511395.11     4242      31.0            3.2           389.16
2025-06  1615491.78       640771.16  462959.67  511760.95     4164      31.7           -2.1           387.97

Nota: los datos cubren de enero a junio de 2025 únicamente, por lo que no es posible calcular YoY. S

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión — credenciales de ejemplo. Usa variables de entorno para las reales, nunca las escribas aquí.
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'TU_CONTRASEÑA_AQUI',  # nunca subas la contraseña real a un repo público
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [ ]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos,
        CASE nombre_evento
            WHEN 'first_visit'      THEN 1
            WHEN 'select_item'      THEN 2
            WHEN 'add_to_cart'      THEN 3
            WHEN 'begin_checkout'   THEN 4
            WHEN 'add_payment_info' THEN 5
            WHEN 'purchase'         THEN 6
        END AS Paso
    FROM events
    GROUP BY nombre_evento
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios_unicos,paso
0,add_payment_info,6250,5
1,add_to_cart,7634,3
2,begin_checkout,7208,4
3,first_visit,7796,1
4,purchase,6240,6
5,select_item,7582,2


In [ ]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
    WITH funnel AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos,
        CASE nombre_evento
            WHEN 'first_visit'      THEN 1
            WHEN 'select_item'      THEN 2
            WHEN 'add_to_cart'      THEN 3
            WHEN 'begin_checkout'   THEN 4
            WHEN 'add_payment_info' THEN 5
            WHEN 'purchase'         THEN 6
        END AS paso
    FROM events
    GROUP BY nombre_evento
)
SELECT
    paso,
    nombre_evento,
    usuarios_unicos,
    LAG(usuarios_unicos) OVER (ORDER BY paso) AS usuarios_paso_anterior,
    usuarios_unicos - LAG(usuarios_unicos) OVER (ORDER BY paso) AS usuarios_perdidos,
    COALESCE(
        ROUND(usuarios_unicos * 100.0 / LAG(usuarios_unicos) OVER (ORDER BY paso), 2),
        100
    ) AS conversion_paso_pct,
    ROUND(
        usuarios_unicos * 100.0 / FIRST_VALUE(usuarios_unicos) OVER (ORDER BY paso),
        2
    ) AS conversion_acumulada_pct
FROM funnel
WHERE paso IS NOT NULL
ORDER BY paso
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,paso,nombre_evento,usuarios_unicos,usuarios_paso_anterior,usuarios_perdidos,conversion_paso_pct,conversion_acumulada_pct
0,1,first_visit,7796,NaN,NaN,100.00,100.00
1,2,select_item,7582,7796.0,-214.0,97.26,97.26
2,3,add_to_cart,7634,7582.0,52.0,100.69,97.92
3,4,begin_checkout,7208,7634.0,-426.0,94.42,92.46
4,5,add_payment_info,6250,7208.0,-958.0,86.71,80.17
5,6,purchase,6240,6250.0,-10.0,99.84,80.04


In [ ]:
cuello = conversion.iloc[1:].sort_values('conversion_paso_pct').iloc[0]
conversion_final = conversion.iloc[-1]['conversion_acumulada_pct']
 
print(f'Tasa de conversión final (first_visit -> purchase): {conversion_final}%')
print(f'\nMayor caída del funnel: {cuello["nombre_evento"]}')
print(f'   - Solo el {cuello["conversion_paso_pct"]}% de los usuarios avanza desde el paso anterior')
print(f'   - Se pierden {abs(int(cuello["usuarios_perdidos"])):,} usuarios en esa etapa')

Tasa de conversión final (first_visit -> purchase): 80.04%

Mayor caída del funnel: add_payment_info
   - Solo el 86.71% de los usuarios avanza desde el paso anterior
   - Se pierden 958 usuarios en esa etapa


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
    SELECT *
    FROM user_activity
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [ ]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''

    WITH cohort AS (
    SELECT 
        TO_CHAR(fecha_registro::DATE, 'YYYY-MM') AS mes_cohorte,
        COUNT(DISTINCT users.id_usuario) AS cantidad_usuarios,
        COUNT(DISTINCT CASE WHEN activo = 1 
        AND dias_despues_registro BETWEEN 1 AND 7 THEN user_activity.id_usuario END) AS retenido_w1,
        COUNT(DISTINCT CASE WHEN activo = 1 
        AND dias_despues_registro BETWEEN 8 AND 14 THEN user_activity.id_usuario END) AS retenido_w2,
        COUNT(DISTINCT CASE WHEN activo = 1 
        AND dias_despues_registro BETWEEN 15 AND 21 THEN user_activity.id_usuario END) AS retenido_w3
    FROM users
    LEFT JOIN user_activity ON users.id_usuario = user_activity.id_usuario
    GROUP BY mes_cohorte
    )
    SELECT
        mes_cohorte,
        cantidad_usuarios,
        retenido_w1,
        retenido_w2,
        retenido_w3,
        ROUND(CAST(retenido_w1 AS NUMERIC) / cantidad_usuarios * 100, 2) AS semana_1,
        ROUND(CAST(retenido_w2 AS NUMERIC) / cantidad_usuarios * 100, 2) AS semana_2,
        ROUND(CAST(retenido_w3 AS NUMERIC) / cantidad_usuarios * 100, 2) AS semana_3
    FROM cohort
    ORDER BY mes_cohorte
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,mes_cohorte,cantidad_usuarios,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01,1627,697,668,656,42.84,41.06,40.32
1,2025-02,1444,611,609,635,42.31,42.17,43.98
2,2025-03,1636,677,705,690,41.38,43.09,42.18
3,2025-04,1606,680,697,663,42.34,43.40,41.28
4,2025-05,1687,695,676,706,41.20,40.07,41.85


In [ ]:
#INTERPRETACION DE COHORTES
print('--- Retención promedio de todas las cohortes ---')
print(f'Semana 1: {cohorte_final["semana_1"].mean():.2f}%')
print(f'Semana 2: {cohorte_final["semana_2"].mean():.2f}%')
print(f'Semana 3: {cohorte_final["semana_3"].mean():.2f}%')
 
caida = (1 - cohorte_final['semana_3'].mean() / cohorte_final['semana_1'].mean()) * 100
print(f'\nCaída de la semana 1 a la semana 3: {caida:.2f}%')
 
mejor = cohorte_final.loc[cohorte_final['semana_1'].idxmax()]
peor = cohorte_final.loc[cohorte_final['semana_1'].idxmin()]
print(f'Mejor cohorte (semana 1): {mejor["mes_cohorte"]} con {mejor["semana_1"]}%')
print(f'Peor cohorte (semana 1):  {peor["mes_cohorte"]} con {peor["semana_1"]}%')

--- Retención promedio de todas las cohortes ---
Semana 1: 42.01%
Semana 2: 41.96%
Semana 3: 41.92%

Caída de la semana 1 a la semana 3: 0.22%
Mejor cohorte (semana 1): 2025-01 con 42.84%
Peor cohorte (semana 1):  2025-05 con 41.2%


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** La modificacion en la UI del checkout no impacta significativamente la tasa de conversion de compra.
   - **H₁ (Hipótesis alternativa):** La modificacion en la UI del checkout impacta la tasa de conversion de compra significativamente.
   
**Test estadístico:** z-test de proporciones
**Nivel de significancia alpha:** 0.05

In [ ]:
#Importar libreria
from statsmodels.stats.proportion import proportions_ztest

#Cargar Dataframe 
checkout_test = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')

#Convertir timestamp a datetime64
checkout_test['timestamp'] = pd.to_datetime(checkout_test['timestamp'])

#Preparacion de test
alpha = 0.05

#Agrupar por variente de cliente convertido solamente
resumen = checkout_test.groupby('variante')['convirtio'].agg(['sum', 'count'])
resumen.columns = ['conversiones', 'usuarios']
resumen['tasa_conversion_pct'] = (resumen['conversiones'] / resumen['usuarios'] * 100).round(2) #agregar columna de tasa_conversion_pct
print(resumen.to_string())
 
p_trat = resumen.loc['tratamiento', 'tasa_conversion_pct']
p_ctrl = resumen.loc['control', 'tasa_conversion_pct']
print(f'\nDiferencia observada: {p_trat - p_ctrl:+.2f} puntos porcentuales')
print(f'Cambio relativo: {(p_trat / p_ctrl - 1) * 100:+.2f}%')

             conversiones  usuarios  tasa_conversion_pct
variante                                                
control               779      4965                15.69
tratamiento           820      5035                16.29

Diferencia observada: +0.60 puntos porcentuales
Cambio relativo: +3.82%


In [ ]:
#Ejecucion del Z-test
conversiones = [resumen.loc['tratamiento', 'conversiones'], resumen.loc['control', 'conversiones']]
totales = [resumen.loc['tratamiento', 'usuarios'], resumen.loc['control', 'usuarios']]
 
stat, p_value = proportions_ztest(count=conversiones, nobs=totales)
 
print(f'Z-statistic: {stat:.4f}')
print(f'P-value:     {p_value:.4f}')
print(f'Alpha:       {alpha}')

Z-statistic: 0.8133
P-value:     0.4161
Alpha:       0.05


In [ ]:
#INTERPRETACION
if p_value < alpha:
    print(f'p-value ({p_value:.4f}) < alpha ({alpha}) → SE RECHAZA H0')
    print('Hay evidencia estadística de que el nuevo diseño del checkout cambia')
    print(f'la tasa de conversión ({p_trat - p_ctrl:+.2f} puntos porcentuales).')
else:
    print(f'p-value ({p_value:.4f}) > alpha ({alpha}) → NO se rechaza H0')
    print('No hay evidencia estadística suficiente para afirmar que el nuevo diseño')
    print('del checkout tenga efecto sobre la tasa de conversión.')
    print('\nOJO: no rechazar H0 no significa demostrar que el')
    print('cambio no sirve. Significa que este experimento no logró detectar un')
    print('efecto. El intervalo de confianza muestra qué tan grande podría ser')
    print('el efecto real sin que el test lo distinga del azar.')

p-value (0.4161) > alpha (0.05) → NO se rechaza H0
No hay evidencia estadística suficiente para afirmar que el nuevo diseño
del checkout tenga efecto sobre la tasa de conversión.

OJO: no rechazar H0 no significa demostrar que el
cambio no sirve. Significa que este experimento no logró detectar un
efecto. El intervalo de confianza muestra qué tan grande podría ser
el efecto real sin que el test lo distinga del azar.


In [ ]:
# Resultados de los pasos 3, 4 y 5: sirven para agregar una hoja de funnel
# y una de retención al dashboard, en vez de solo describirlos en el notebook.
cohorte_final.to_csv('cohorte_final.csv', index=False)
resumen.reset_index().to_csv('experimento_ab.csv', index=False)
 
print('Archivos listos para Tableau')

Archivos listos para Tableau


In [ ]:
# Datos para corroborar que el Dashboard arroje los resultados correctos
 
print('=' * 40)
print('NÚMEROS DE CONTROL')
print('=' * 40)
print(f'Pedidos:            {orders["id_pedido"].nunique():,}')
print(f'Revenue total:      ${revenue_total:,.2f}')
print(f'Costo de producto:  ${costo_producto:,.2f}')
print(f'Gasto marketing:    ${gasto_marketing:,.2f}')
print(f'Profit:             ${profit:,.2f}')
print(f'Margen neto:        {profit / revenue_total * 100:.2f}%')
print(f'Ticket promedio:    ${ticket_promedio:,.2f}')
print(f'Unidades por orden: {cantidad_promedio:.2f}')
print(f'ROAS:               {revenue_total / gasto_marketing:.2f}x')
print('=' * 40)

NÚMEROS DE CONTROL
Pedidos:            24,936
Revenue total:      $9,622,281.56
Costo de producto:  $3,828,869.01
Gasto marketing:    $2,694,664.43
Profit:             $3,098,748.12
Margen neto:        32.20%
Ticket promedio:    $385.88
Unidades por orden: 1.50
ROAS:               3.57x


---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---